# CFPB Complaint Data — EDA & Preprocessing

Task 1: explore the raw CFPB complaint export, then filter and clean it for the RAG pipeline.

**Note:** the data load path below assumes this notebook lives in a `notebooks/` folder and the raw CSV lives in a sibling `data/raw/` folder (i.e. `../data/raw/complaints.csv`). Adjust the path if your folder layout differs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path

sns.set_style('whitegrid')

# Load data
# FIX: original path was '..data/raw/complaints.csv' (typo: two dots, no slash)
# Correct relative path is '../data/raw/complaints.csv'
candidate_paths = [
    '../data/raw/complaints.csv',
    '../data/complaints.csv',
    'data/raw/complaints.csv',
    'complaints.csv',
    '../complaints.csv'
]

for p in candidate_paths:
    if Path(p).exists():
        try:
            df = pd.read_csv(p, low_memory=False)
        except PermissionError:
            print(f"Permission denied reading {p}; trying next candidate.")
            continue
        except Exception as exc:
            print(f"Unable to read {p}: {exc}; trying next candidate.")
            continue
        print(f"Loaded data from: {p}")
        break
        print(f"Loaded data from: {p}")
        break
else:
    raise FileNotFoundError(
        "Could not find 'complaints.csv'. Checked paths:\n" +
        "\n".join(candidate_paths) +
        "\n\nPut the file in ../data/raw/complaints.csv or update the path."
    )

# Quick inspection
print(df.info())
print(df.head())

PermissionError: [Errno 13] Permission denied: 'complaints.csv'

## 1. Exploratory Data Analysis

Covers: product distribution, narrative length distribution, and narrative presence counts.

In [ ]:
# --- Distribution of complaints across products ---
product_counts = df['Product'].value_counts()
print(f"Total unique products in raw data: {df['Product'].nunique()}")
print(product_counts)

plt.figure(figsize=(10, 6))
product_counts.plot(kind='bar', color='#4C72B0')
plt.title('Complaint Distribution by Product (All Products, Raw Data)')
plt.ylabel('Number of complaints')
plt.xlabel('Product')
plt.xticks(rotation=75, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/product_distribution_all.png', dpi=150)
plt.show()

In [ ]:
# --- Complaints with vs. without a narrative ---
has_narrative = df['Consumer complaint narrative'].notna().sum()
no_narrative = df['Consumer complaint narrative'].isna().sum()
total = len(df)

print(f"Total complaints: {total:,}")
print(f"With narrative:    {has_narrative:,} ({has_narrative/total:.1%})")
print(f"Without narrative: {no_narrative:,} ({no_narrative/total:.1%})")

plt.figure(figsize=(5, 5))
plt.pie(
    [has_narrative, no_narrative],
    labels=['Has narrative', 'No narrative'],
    autopct='%1.1f%%',
    colors=['#55A868', '#CCCCCC'],
    startangle=90
)
plt.title('Narrative Availability (Raw Data)')
plt.tight_layout()
plt.savefig('../reports/figures/narrative_availability.png', dpi=150)
plt.show()

In [ ]:
# --- Narrative length (word count) distribution ---
narratives = df['Consumer complaint narrative'].dropna()
word_counts = narratives.str.split().str.len()

print(word_counts.describe())
print(f"\nVery short narratives (<10 words): {(word_counts < 10).sum():,}")
print(f"Very long narratives (>1000 words): {(word_counts > 1000).sum():,}")

plt.figure(figsize=(10, 6))
plt.hist(word_counts.clip(upper=1500), bins=60, color='#4C72B0', edgecolor='white', linewidth=0.3)
plt.axvline(word_counts.median(), color='#C44E52', linestyle='--',
            label=f'Median = {word_counts.median():.0f} words')
plt.title('Narrative Length Distribution (word count, clipped at 1500)')
plt.xlabel('Word count')
plt.ylabel('Number of complaints')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/figures/narrative_length_distribution.png', dpi=150)
plt.show()

## 2. Filtering

Retain only the four target products and drop rows with no narrative.

In [ ]:
# --- Filter to target products ---
target_products = [
    'Credit card',
    'Personal loan',
    'Checking or savings account',
    'Money transfer, virtual currency, or money service'
]

print('Product labels actually present in raw data (for comparison against target list):')
print(sorted(df['Product'].unique()))

before_count = len(df)
df_filtered = df[df['Product'].isin(target_products)].copy()
print(f"\nRows before product filter: {before_count:,}")
print(f"Rows after product filter:  {len(df_filtered):,}")

# Sanity check: warn if a target product had zero matches (likely a label mismatch)
matched_products = df_filtered['Product'].unique()
missing_targets = [p for p in target_products if p not in matched_products]
if missing_targets:
    print(f"\nWARNING: these target products had 0 matches — check exact label spelling in raw data: {missing_targets}")

# Drop rows with empty narrative
before_narrative_drop = len(df_filtered)
df_filtered = df_filtered.dropna(subset=['Consumer complaint narrative'])
print(f"\nRows before narrative-empty filter: {before_narrative_drop:,}")
print(f"Rows after narrative-empty filter:  {len(df_filtered):,}")

print("\nFinal product distribution after filtering:")
print(df_filtered['Product'].value_counts())

## 3. Text Cleaning

Lowercase, strip boilerplate opening phrases, remove special characters.

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Lowercase
    text = text.lower()
    # Remove a common boilerplate opening phrase
    text = re.sub(r"i am writing to file a complaint against.*?\.", "", text)
    # Remove other common templated openers (extend as you observe more in your data)
    text = re.sub(r"this complaint is regarding.*?\.", "", text)
    text = re.sub(r"i would like to file a complaint.*?\.", "", text)
    # Remove special characters, keep basic punctuation
    text = re.sub(r"[^a-zA-Z0-9\s\.,!?]", "", text)
    # Collapse repeated whitespace left by removals
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df_filtered['cleaned_narrative'] = df_filtered['Consumer complaint narrative'].apply(clean_text)

# Quick before/after check on a sample row
sample_idx = df_filtered.index[0]
print('BEFORE:', df_filtered.loc[sample_idx, 'Consumer complaint narrative'][:300])
print('\nAFTER: ', df_filtered.loc[sample_idx, 'cleaned_narrative'][:300])

In [ ]:
# --- Save cleaned/filtered dataset ---
import os
os.makedirs('../data', exist_ok=True)
output_path = '../data/filtered_complaints.csv'
df_filtered.to_csv(output_path, index=False)
print(f"Saved {len(df_filtered):,} cleaned records to {output_path}")